In [1]:
from glob import glob
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [2]:
files = sorted(glob("data/baci/BACI_HS17_Y*V202501.csv"))

df_list = [pd.read_csv(f) for f in files]
df = pd.concat(df_list, ignore_index=True)
# df.to_csv("data/concat_df.csv")

In [3]:
df = df.rename(columns={
    "t": "year",
    "i": "exporter",
    "j": "importer",
    "k": "hs6",
    "v": "value",
    "q": "quantity"
})
df.head(5)

,year,exporter,importer,hs6,value,quantity
0,2017,4,12,130120,5.946,1.400
1,2017,4,12,130190,5.125,2.320
2,2017,4,12,401031,0.087,0.002
3,2017,4,12,853890,0.303,0.019
4,2017,4,36,71320,3.538,2.446


In [4]:
df["hs6"] = df["hs6"].astype(str).str.zfill(6)# добавляем ведущий ноль, если нет
df = df[df["hs6"].str.startswith("09")]
df["hs2"] = df["hs6"].str[:2]

df_agg = (
    df.groupby(["year", "exporter", "importer", "hs2"], as_index=False)
      .agg({"value": "sum", "quantity": "sum"})
)
df_agg.head(5)

,year,exporter,importer,hs2,value,quantity
0,2017,4,36,09,20.641,1.119
1,2017,4,40,09,2.375,0.013
2,2017,4,48,09,0.664,0.033
3,2017,4,56,09,0.102,0.002
4,2017,4,124,09,43.211,1.400


In [5]:
country_map = pd.read_csv("data/baci/country_codes_V202501.csv") 

#для экспортера
df_agg = df_agg.merge(
    country_map[["country_code", "country_name", "country_iso3"]],
    left_on="exporter",
    right_on="country_code",
    how="left"
)
df_agg["exporter_iso3"] = df_agg["country_iso3"]
df_agg["exporter"] = df_agg["country_name"]
df_agg = df_agg.drop(columns=["country_code", "country_name", "country_iso3"])

#для импортера
df_agg = df_agg.merge(
    country_map[["country_code", "country_name", "country_iso3"]],
    left_on="importer",
    right_on="country_code",
    how="left"
)

df_agg["importer_iso3"] = df_agg["country_iso3"]
df_agg["importer"] = df_agg["country_name"]
df_agg = df_agg.drop(columns=["country_code", "country_name", "country_iso3"])

df_agg.head(5)

,year,exporter,importer,hs2,value,quantity,exporter_iso3,importer_iso3
0,2017,Afghanistan,Australia,09,20.641,1.119,AFG,AUS
1,2017,Afghanistan,Austria,09,2.375,0.013,AFG,AUT
2,2017,Afghanistan,Bahrain,09,0.664,0.033,AFG,BHR
3,2017,Afghanistan,Belgium,09,0.102,0.002,AFG,BEL
4,2017,Afghanistan,Canada,09,43.211,1.400,AFG,CAN


In [6]:
#усекаем по России
df_agg = df_agg[(df_agg["exporter"] == "Russian Federation") | (df_agg["importer"] == "Russian Federation")]

#добавляем партнера
df_agg["partner"] = df_agg.apply(
    lambda r: r["importer"] if r["exporter"]== "Russian Federation" else r["exporter"],
    axis=1
)

df_agg["partner_iso3"] = df_agg.apply(
    lambda r: r["importer_iso3"] if r["exporter_iso3"]== "RUS" else r["exporter_iso3"],
    axis=1
)

#допавляем направление
df_agg["direction"] = df_agg["exporter"].apply(
    lambda x: "export" if x == "Russian Federation" else "import"
)

#добавляем цену
df_agg["price"] = df_agg["value"] / df_agg["quantity"]

df_agg.head(5)

,year,exporter,importer,hs2,value,quantity,exporter_iso3,importer_iso3,partner,partner_iso3,direction,price
60,2017,Albania,Russian Federation,09,10.014,5.001,ALB,RUS,Albania,ALB,import,2.002400
122,2017,Azerbaijan,Russian Federation,09,5144.972,1037.844,AZE,RUS,Azerbaijan,AZE,import,4.957365
175,2017,Argentina,Russian Federation,09,2431.309,1605.177,ARG,RUS,Argentina,ARG,import,1.514667
259,2017,Australia,Russian Federation,09,5.605,0.237,AUS,RUS,Australia,AUS,import,23.649789
366,2017,Austria,Russian Federation,09,19119.567,821.902,AUT,RUS,Austria,AUT,import,23.262587


ВВП(current US$)

In [7]:
df_gpd = pd.read_csv("data/gdp.csv")
df_gpd.head(3)

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,Unnamed: 69
0,Aruba,ABW,GDP (current US$),NY.GDP.MKTP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,2.983635e+09,3.092429e+09,3.276184e+09,3.395799e+09,2.481857e+09,2.929447e+09,3.279344e+09,3.648573e+09,NaN,NaN
1,Africa Eastern and Southern,AFE,GDP (current US$),NY.GDP.MKTP.CD,2.420993e+10,2.496326e+10,2.707802e+10,3.177483e+10,3.028492e+10,3.381219e+10,...,8.289612e+11,9.730251e+11,1.012291e+12,1.009747e+12,9.334072e+11,1.085605e+12,1.191639e+12,1.133818e+12,1.205974e+12,NaN
2,Afghanistan,AFG,GDP (current US$),NY.GDP.MKTP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,1.811657e+10,1.875346e+10,1.805322e+10,1.879944e+10,1.995593e+10,1.426000e+10,1.449724e+10,1.715223e+10,NaN,NaN


In [8]:
df_gpd = df_gpd.drop(columns=["Indicator Name", "Indicator Code", "Unnamed: 69"])
years = [str(y) for y in range(2017, 2024)]
df_gpd = df_gpd[["Country Name", "Country Code"] + years]

gdp_long = df_gpd.melt(
    id_vars=["Country Name", "Country Code"],
    value_vars=years,
    var_name="year",
    value_name="gdp_usd"
)
gdp_long["year"] = gdp_long["year"].astype(int)
gdp_long.head(3)

,Country Name,Country Code,year,gdp_usd
0,Aruba,ABW,2017,3.092429e+09
1,Africa Eastern and Southern,AFE,2017,9.730251e+11
2,Afghanistan,AFG,2017,1.875346e+10


In [9]:
# ВВП партнера
gdp_merged = df_agg.merge(
    gdp_long,
    left_on=["partner_iso3", "year"],
    right_on=["Country Code", "year"],
    how="left"
).rename(columns={"gdp_usd": "gdp_partner"}).drop(columns=["Country Code", "Country Name"])

# ВВП РФ
russia_gdp = gdp_long[gdp_long["Country Code"] == "RUS"][["year", "gdp_usd"]].rename(columns={"gdp_usd": "gdp_russia"})

gdp_merged = gdp_merged.merge(
    russia_gdp,
    on="year",
    how="left"
)

gdp_merged.head(3)

,year,exporter,importer,hs2,value,quantity,exporter_iso3,importer_iso3,partner,partner_iso3,direction,price,gdp_partner,gdp_russia
0,2017,Albania,Russian Federation,09,10.014,5.001,ALB,RUS,Albania,ALB,import,2.002400,1.325827e+10,1.574199e+12
1,2017,Azerbaijan,Russian Federation,09,5144.972,1037.844,AZE,RUS,Azerbaijan,AZE,import,4.957365,4.086663e+10,1.574199e+12
2,2017,Argentina,Russian Federation,09,2431.309,1605.177,ARG,RUS,Argentina,ARG,import,1.514667,6.436284e+11,1.574199e+12


Индикаторы CEPII(расстояние, наличие общей границы, языка)

In [10]:
cepii_df = pd.read_excel("data/dist_cepii.xls")

cepii_merged = gdp_merged.merge(
    cepii_df[['iso_o', 'iso_d', 'dist', 'contig', 'comlang_off']],
    left_on=['exporter_iso3', 'importer_iso3'],  # exporter → iso_o, importer → iso_d
    right_on=['iso_o', 'iso_d'],
    how='left'
).drop(columns=["iso_o", "iso_d"])

cepii_merged.head(3)

,year,exporter,importer,hs2,value,quantity,exporter_iso3,importer_iso3,partner,partner_iso3,direction,price,gdp_partner,gdp_russia,dist,contig,comlang_off
0,2017,Albania,Russian Federation,09,10.014,5.001,ALB,RUS,Albania,ALB,import,2.002400,1.325827e+10,1.574199e+12,2063.678,0.0,0.0
1,2017,Azerbaijan,Russian Federation,09,5144.972,1037.844,AZE,RUS,Azerbaijan,AZE,import,4.957365,4.086663e+10,1.574199e+12,1930.747,1.0,0.0
2,2017,Argentina,Russian Federation,09,2431.309,1605.177,ARG,RUS,Argentina,ARG,import,1.514667,6.436284e+11,1.574199e+12,13505.360,0.0,0.0


CPI(Инфляция, потребительские цены в годовом исчислении %)

In [11]:
df_cpi = pd.read_csv("data/cpi.csv", skiprows=4)
df_cpi.head(1)

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,Unnamed: 69
0,Aruba,ABW,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.931196,-1.028282,3.626041,4.257462,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
df_cpi = df_cpi.drop(columns=["Indicator Name", "Indicator Code", "Unnamed: 69"])
years = [str(y) for y in range(2017, 2024)]
df_cpi = df_cpi[["Country Name", "Country Code"] + years]

cpi_long = df_cpi.melt(
    id_vars=["Country Name", "Country Code"],
    value_vars=years,
    var_name="year",
    value_name="cpi_usd"
)
cpi_long["year"] = cpi_long["year"].astype(int)

cpi_long.head(3)

,Country Name,Country Code,year,cpi_usd
0,Aruba,ABW,2017,-1.028282
1,Africa Eastern and Southern,AFE,2017,6.399343
2,Afghanistan,AFG,2017,4.975952


In [13]:
# Инфляция партнера
cpi_merged = cepii_merged.merge(
    cpi_long,
    left_on=["partner_iso3", "year"],
    right_on=["Country Code", "year"],
    how="left"
).rename(columns={"cpi_usd": "cpi_partner"}).drop(columns=["Country Code", "Country Name"])

# Инфляция РФ
russia_сpi = cpi_long[cpi_long["Country Code"] == "RUS"][["year", "cpi_usd"]].rename(columns={"cpi_usd": "cpi_russia"})

cpi_merged = cpi_merged.merge(
    russia_сpi,
    on="year",
    how="left"
)

cpi_merged.head(3)

,year,exporter,importer,hs2,value,quantity,exporter_iso3,importer_iso3,partner,partner_iso3,direction,price,gdp_partner,gdp_russia,dist,contig,comlang_off,cpi_partner,cpi_russia
0,2017,Albania,Russian Federation,09,10.014,5.001,ALB,RUS,Albania,ALB,import,2.002400,1.325827e+10,1.574199e+12,2063.678,0.0,0.0,1.986661,3.683329
1,2017,Azerbaijan,Russian Federation,09,5144.972,1037.844,AZE,RUS,Azerbaijan,AZE,import,4.957365,4.086663e+10,1.574199e+12,1930.747,1.0,0.0,12.935918,3.683329
2,2017,Argentina,Russian Federation,09,2431.309,1605.177,ARG,RUS,Argentina,ARG,import,1.514667,6.436284e+11,1.574199e+12,13505.360,0.0,0.0,NaN,3.683329


Обменный курс(LCU за доллар США, средний за период)

In [14]:
df_ex_rate = pd.read_csv("data/ex_rate.csv", skiprows=4)
df_ex_rate.head(1)

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,Unnamed: 69
0,Aruba,ABW,"Official exchange rate (LCU per US$, period av...",PA.NUS.FCRF,NaN,NaN,NaN,NaN,NaN,NaN,...,1.79,1.79,1.79,1.79,1.79,1.79,1.79,1.79,1.79,NaN


In [15]:
df_ex_rate = df_ex_rate.drop(columns=["Indicator Name", "Indicator Code", "Unnamed: 69"])
years = [str(y) for y in range(2017, 2024)]
df_ex_rate = df_ex_rate[["Country Name", "Country Code"] + years]

ex_rate_long = df_ex_rate.melt(
    id_vars=["Country Name", "Country Code"],
    value_vars=years,
    var_name="year",
    value_name="ex_rate"
)
ex_rate_long["year"] = ex_rate_long["year"].astype(int)
ex_rate_long.head(3)

,Country Name,Country Code,year,ex_rate
0,Aruba,ABW,2017,1.790000
1,Africa Eastern and Southern,AFE,2017,NaN
2,Afghanistan,AFG,2017,68.026904


In [16]:
# Курс партнера
ex_rate_merged = cpi_merged.merge(
    ex_rate_long,
    left_on=["partner_iso3", "year"],
    right_on=["Country Code", "year"],
    how="left"
).rename(columns={"ex_rate": "ex_rate_partner"}).drop(columns=["Country Code", "Country Name"])

# Курс рубля
russia_ex_rate = ex_rate_long[ex_rate_long["Country Code"] == "RUS"][["year", "ex_rate"]].rename(columns={"ex_rate": "ex_rate_russia"})
ex_rate_merged = ex_rate_merged.merge(
    russia_ex_rate,
    on="year",
    how="left"
)

# Двусторонний курс
ex_rate_merged["bilateral_er"] = (
    ex_rate_merged["ex_rate_partner"] / ex_rate_merged["ex_rate_russia"]
)

ex_rate_merged.head(3)

,year,exporter,importer,hs2,value,quantity,exporter_iso3,importer_iso3,partner,partner_iso3,...,gdp_partner,gdp_russia,dist,contig,comlang_off,cpi_partner,cpi_russia,ex_rate_partner,ex_rate_russia,bilateral_er
0,2017,Albania,Russian Federation,09,10.014,5.001,ALB,RUS,Albania,ALB,...,1.325827e+10,1.574199e+12,2063.678,0.0,0.0,1.986661,3.683329,119.100000,58.342801,2.041383
1,2017,Azerbaijan,Russian Federation,09,5144.972,1037.844,AZE,RUS,Azerbaijan,AZE,...,4.086663e+10,1.574199e+12,1930.747,1.0,0.0,12.935918,3.683329,1.721155,58.342801,0.029501
2,2017,Argentina,Russian Federation,09,2431.309,1605.177,ARG,RUS,Argentina,ARG,...,6.436284e+11,1.574199e+12,13505.360,0.0,0.0,NaN,3.683329,16.562707,58.342801,0.283886


IIP(Индекс промышленного производства)

In [17]:
df_iip = pd.read_csv("data/unido_iip/data_C.csv") 
df_iip.head(1)

,Year,Country,CountryCode,Variable,VariableCode,UnconsolidatedVariableCode,UnconsolidatedVariableName,ActivityCode,Activity,ActivityCombination,Value,UnitType,Metadata
0,2017,Albania,8,Original index,52,52,Original index,C,Total manufacturing,C,100.82,I,NaN


In [18]:
df_iip = df_iip.merge(
    country_map,
    left_on="CountryCode",
    right_on="country_code",
    how="left"
).loc[:, ["Year", "Country", "CountryCode", "country_iso3", "Value"]]
df_iip.rename(columns={"Year": "year"}, inplace=True)
df_iip.head(1)

,year,Country,CountryCode,country_iso3,Value
0,2017,Albania,8,ALB,100.82


In [19]:
iip_merged = ex_rate_merged.merge(
    df_iip,
    left_on=["partner_iso3", "year"],
    right_on=["country_iso3", "year"],
    how="left"
).rename(columns={"Value": "iip_partner"}).drop(columns=["Country", "CountryCode", "country_iso3"])

russia_iip = df_iip[df_iip["country_iso3"] == "RUS"][["year", "Value"]].rename(columns={"Value": "iip_russia"})

iip_merged = iip_merged.merge(
    russia_iip,
    on="year",
    how="left"
)

iip_merged.head(3)

,year,exporter,importer,hs2,value,quantity,exporter_iso3,importer_iso3,partner,partner_iso3,...,dist,contig,comlang_off,cpi_partner,cpi_russia,ex_rate_partner,ex_rate_russia,bilateral_er,iip_partner,iip_russia
0,2017,Albania,Russian Federation,09,10.014,5.001,ALB,RUS,Albania,ALB,...,2063.678,0.0,0.0,1.986661,3.683329,119.100000,58.342801,2.041383,100.82,106.88
1,2017,Azerbaijan,Russian Federation,09,5144.972,1037.844,AZE,RUS,Azerbaijan,AZE,...,1930.747,1.0,0.0,12.935918,3.683329,1.721155,58.342801,0.029501,99.07,106.88
2,2017,Argentina,Russian Federation,09,2431.309,1605.177,ARG,RUS,Argentina,ARG,...,13505.360,0.0,0.0,NaN,3.683329,16.562707,58.342801,0.283886,97.79,106.88


Недружественные страны

In [20]:
unfriendly_iso3 = {#Так сказал ЧАТ
    "AUT", "BEL", "BGR", "HRV", "CYP", "CZE", "DNK", "EST", "FIN", "FRA",
    "DEU", "GRC", "HUN", "IRL", "ITA", "LVA", "LTU", "LUX", "MLT", "NLD",
    "POL", "PRT", "ROU", "SVK", "SVN", "ESP", "SWE","USA", "CAN", "GBR", "AUS", "NZL","JPN", "KOR", "TWN", "SGP", "CHE", "NOR", "ISL", "LIE", "MCO", "ALB", "MNE", "UKR",
}

iip_merged["unfriendly"] = iip_merged['partner_iso3'].isin(unfriendly_iso3).astype(int)

iip_merged.head(3)

,year,exporter,importer,hs2,value,quantity,exporter_iso3,importer_iso3,partner,partner_iso3,...,contig,comlang_off,cpi_partner,cpi_russia,ex_rate_partner,ex_rate_russia,bilateral_er,iip_partner,iip_russia,unfriendly
0,2017,Albania,Russian Federation,09,10.014,5.001,ALB,RUS,Albania,ALB,...,0.0,0.0,1.986661,3.683329,119.100000,58.342801,2.041383,100.82,106.88,1
1,2017,Azerbaijan,Russian Federation,09,5144.972,1037.844,AZE,RUS,Azerbaijan,AZE,...,1.0,0.0,12.935918,3.683329,1.721155,58.342801,0.029501,99.07,106.88,0
2,2017,Argentina,Russian Federation,09,2431.309,1605.177,ARG,RUS,Argentina,ARG,...,0.0,0.0,NaN,3.683329,16.562707,58.342801,0.283886,97.79,106.88,0


Санкционность

In [21]:
first_sanction_year = {#Тоже сказал ЧАТ
    # ----- основная первая волна (2014 — Крым) -----
    "USA": 2014, "CAN": 2014, "GBR": 2014, "FRA": 2014, "DEU": 2014, "ITA": 2014,
    "ESP": 2014, "NLD": 2014, "BEL": 2014, "SWE": 2014, "DNK": 2014, "FIN": 2014,
    "POL": 2014, "PRT": 2014, "GRC": 2014, "IRL": 2014, "AUT": 2014, "CZE": 2014,
    "SVK": 2014, "SVN": 2014, "HRV": 2014, "BGR": 2014, "ROU": 2014, "LVA": 2014,
    "LTU": 2014, "EST": 2014, "LUX": 2014, "MLT": 2014,
    "AUS": 2014, "JPN": 2014, "NOR": 2014, "ISL": 2014, "CAN": 2014,
    # note: UK included above as GBR (was in EU in 2014)
    # Switzerland: *joined major 2022 packages* (see note) — keep as 2022 below

    # ----- присоединившиеся/впервые в 2022 -----
    "KOR": 2022, "SGP": 2022, "TWN": 2022, "NZL": 2022,
    "ALB": 2022, "MNE": 2022, "MKD": 2022, "XKX": 2022, "MCO": 2022, "LIE": 2022,
    "CHE": 2022,   # Switzerland: de-facto joined many 2022 measures (treated as 2022 here)

    # ----- страны, не вводившие (или не в общих западных пакетах до 2023) -----
    "CHN": None, "IND": None, "TUR": None, "BRA": None, "SAU": None, "ARE": None,
    "ISR": None, "EGY": None, "KAZ": None, "UZB": None, "KGZ": None, "TJK": None,
    "ARM": None, "AZE": None, "MYS": None, "IDN": None, "VNM": None, "THA": None,
    "MEX": None, "ZAF": None, "ARG": None, "PER": None
}

years = list(range(2017, 2024))

def get_sanction_flag(row, first_san_dict):
    iso = row['partner_iso3']
    y = int(row['year'])
    first = first_san_dict.get(iso, None)
    if first is None:
        return 0
    return 1 if y >= first else 0

iip_merged['sanction'] = iip_merged.apply(lambda r: get_sanction_flag(r, first_sanction_year), axis=1)

iip_merged.head(3)

,year,exporter,importer,hs2,value,quantity,exporter_iso3,importer_iso3,partner,partner_iso3,...,comlang_off,cpi_partner,cpi_russia,ex_rate_partner,ex_rate_russia,bilateral_er,iip_partner,iip_russia,unfriendly,sanction
0,2017,Albania,Russian Federation,09,10.014,5.001,ALB,RUS,Albania,ALB,...,0.0,1.986661,3.683329,119.100000,58.342801,2.041383,100.82,106.88,1,0
1,2017,Azerbaijan,Russian Federation,09,5144.972,1037.844,AZE,RUS,Azerbaijan,AZE,...,0.0,12.935918,3.683329,1.721155,58.342801,0.029501,99.07,106.88,0,0
2,2017,Argentina,Russian Federation,09,2431.309,1605.177,ARG,RUS,Argentina,ARG,...,0.0,NaN,3.683329,16.562707,58.342801,0.283886,97.79,106.88,0,0


Политические союзы, в которые входят страны-партнеры

In [22]:
# Евросоюз
eu_countries_iso3 = {
    "AUT": 1,"BEL": 1,"BGR": 1,"HRV": 1,"CYP": 1,"CZE": 1,"DNK": 1,"EST": 1,"FIN": 1,"FRA": 1,"DEU": 1,"GRC": 1,"HUN": 1,"IRL": 1,"ITA": 1,"LVA": 1,"LTU": 1,"LUX": 1,"MLT": 1,"NLD": 1,"POL": 1,"PRT": 1, "ROU": 1,"SVK": 1,"SVN": 1,"ESP": 1,"SWE": 1
}

# Для GBR с 2020 года исключаем
eu_members_by_year = {}
for year in range(2017, 2024):
    if year < 2020:
        eu_members_by_year[year] = list(eu_countries_iso3.keys()) + ["GBR"]
    else:
        eu_members_by_year[year] = list(eu_countries_iso3.keys())
        
eu_rows = []
for year, members in eu_members_by_year.items():
    for iso3 in members:
        eu_rows.append({"year": year, "partner_iso3": iso3, "eu_member": 1})

df_eu = pd.DataFrame(eu_rows)

eu_member_merged = iip_merged.merge(
    df_eu,
    on=["year", "partner_iso3"], 
    how="left")
eu_member_merged["eu_member"] = eu_member_merged["eu_member"].fillna(0).astype(int)

eu_member_merged.head(3)

,year,exporter,importer,hs2,value,quantity,exporter_iso3,importer_iso3,partner,partner_iso3,...,cpi_partner,cpi_russia,ex_rate_partner,ex_rate_russia,bilateral_er,iip_partner,iip_russia,unfriendly,sanction,eu_member
0,2017,Albania,Russian Federation,09,10.014,5.001,ALB,RUS,Albania,ALB,...,1.986661,3.683329,119.100000,58.342801,2.041383,100.82,106.88,1,0,0
1,2017,Azerbaijan,Russian Federation,09,5144.972,1037.844,AZE,RUS,Azerbaijan,AZE,...,12.935918,3.683329,1.721155,58.342801,0.029501,99.07,106.88,0,0,0
2,2017,Argentina,Russian Federation,09,2431.309,1605.177,ARG,RUS,Argentina,ARG,...,NaN,3.683329,16.562707,58.342801,0.283886,97.79,106.88,0,0,0


In [23]:
#ЕАЭС
eaes_countries = ["ARM", "BLR", "KAZ", "KGZ", "RUS"]
eu_member_merged['eaes_member'] = eu_member_merged['partner_iso3'].isin(eaes_countries).astype(int)

eu_member_merged.head(3)

,year,exporter,importer,hs2,value,quantity,exporter_iso3,importer_iso3,partner,partner_iso3,...,cpi_russia,ex_rate_partner,ex_rate_russia,bilateral_er,iip_partner,iip_russia,unfriendly,sanction,eu_member,eaes_member
0,2017,Albania,Russian Federation,09,10.014,5.001,ALB,RUS,Albania,ALB,...,3.683329,119.100000,58.342801,2.041383,100.82,106.88,1,0,0,0
1,2017,Azerbaijan,Russian Federation,09,5144.972,1037.844,AZE,RUS,Azerbaijan,AZE,...,3.683329,1.721155,58.342801,0.029501,99.07,106.88,0,0,0,0
2,2017,Argentina,Russian Federation,09,2431.309,1605.177,ARG,RUS,Argentina,ARG,...,3.683329,16.562707,58.342801,0.283886,97.79,106.88,0,0,0,0


In [24]:
# БРИКС
brics_countries = ["BRA", "RUS", "IND", "CHN", "ZAF"]
eu_member_merged['brics_member'] = eu_member_merged['partner_iso3'].isin(brics_countries).astype(int)

eu_member_merged.head(3)

,year,exporter,importer,hs2,value,quantity,exporter_iso3,importer_iso3,partner,partner_iso3,...,ex_rate_partner,ex_rate_russia,bilateral_er,iip_partner,iip_russia,unfriendly,sanction,eu_member,eaes_member,brics_member
0,2017,Albania,Russian Federation,09,10.014,5.001,ALB,RUS,Albania,ALB,...,119.100000,58.342801,2.041383,100.82,106.88,1,0,0,0,0
1,2017,Azerbaijan,Russian Federation,09,5144.972,1037.844,AZE,RUS,Azerbaijan,AZE,...,1.721155,58.342801,0.029501,99.07,106.88,0,0,0,0,0
2,2017,Argentina,Russian Federation,09,2431.309,1605.177,ARG,RUS,Argentina,ARG,...,16.562707,58.342801,0.283886,97.79,106.88,0,0,0,0,0


In [25]:
# СНГ
cis_countries = [
    "ARM", "AZE", "BLR", "KAZ", "KGZ", "MDA", "RUS", "TJK", "TKM", "UZB"
]
eu_member_merged['cis_member'] = eu_member_merged['partner_iso3'].isin(cis_countries).astype(int)

eu_member_merged.head(3)

,year,exporter,importer,hs2,value,quantity,exporter_iso3,importer_iso3,partner,partner_iso3,...,ex_rate_russia,bilateral_er,iip_partner,iip_russia,unfriendly,sanction,eu_member,eaes_member,brics_member,cis_member
0,2017,Albania,Russian Federation,09,10.014,5.001,ALB,RUS,Albania,ALB,...,58.342801,2.041383,100.82,106.88,1,0,0,0,0,0
1,2017,Azerbaijan,Russian Federation,09,5144.972,1037.844,AZE,RUS,Azerbaijan,AZE,...,58.342801,0.029501,99.07,106.88,0,0,0,0,0,1
2,2017,Argentina,Russian Federation,09,2431.309,1605.177,ARG,RUS,Argentina,ARG,...,58.342801,0.283886,97.79,106.88,0,0,0,0,0,0


In [26]:
# net_export
df_net = eu_member_merged.copy()

imp = df_net[df_net["importer"] == "Russian Federation"][["year", "partner_iso3", "value"]]
imp = imp.rename(columns={"value": "import_value"})

df_net = df_net.merge(imp, on=["year", "partner_iso3"], how="left")

df_net["net_export"] = None

exp_mask = df_net["exporter"] == "Russian Federation"
df_net.loc[exp_mask, "net_export"] = df_net.loc[exp_mask, "value"] - df_net.loc[exp_mask, "import_value"].fillna(0)

df_net
# можно удалить import_value, если не нужна - для проверки
# df_net.drop(columns=["import_value"], inplace=True)

,year,exporter,importer,hs2,value,quantity,exporter_iso3,importer_iso3,partner,partner_iso3,...,iip_partner,iip_russia,unfriendly,sanction,eu_member,eaes_member,brics_member,cis_member,import_value,net_export
0,2017,Albania,Russian Federation,09,10.014,5.001,ALB,RUS,Albania,ALB,...,100.82,106.88,1,0,0,0,0,0,10.014,None
1,2017,Azerbaijan,Russian Federation,09,5144.972,1037.844,AZE,RUS,Azerbaijan,AZE,...,99.07,106.88,0,0,0,0,0,1,5144.972,None
2,2017,Argentina,Russian Federation,09,2431.309,1605.177,ARG,RUS,Argentina,ARG,...,97.79,106.88,0,0,0,0,0,0,2431.309,None
3,2017,Australia,Russian Federation,09,5.605,0.237,AUS,RUS,Australia,AUS,...,98.86,106.88,1,1,0,0,0,0,5.605,None
4,2017,Austria,Russian Federation,09,19119.567,821.902,AUT,RUS,Austria,AUT,...,107.46,106.88,1,1,1,0,0,0,19119.567,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1330,2023,Egypt,Russian Federation,09,157.478,33.442,EGY,RUS,Egypt,EGY,...,98.15,135.97,0,0,0,0,0,0,157.478,None
1331,2023,United Kingdom,Russian Federation,09,1179.479,92.183,GBR,RUS,United Kingdom,GBR,...,110.70,135.97,1,1,0,0,0,0,1179.479,None
1332,2023,United Rep. of Tanzania,Russian Federation,09,5823.626,2604.478,TZA,RUS,United Rep. of Tanzania,TZA,...,163.64,135.97,0,0,0,0,0,0,5823.626,None
1333,2023,USA,Russian Federation,09,8.300,0.373,USA,RUS,USA,USA,...,99.90,135.97,1,1,0,0,0,0,8.300,None


Что получилось

In [27]:
final = df_net[["year", "hs2", "exporter", "importer","exporter_iso3","importer_iso3","partner","partner_iso3","direction","value","quantity","price","net_export","gdp_partner","gdp_russia","cpi_partner","cpi_russia","ex_rate_partner","ex_rate_russia","bilateral_er","iip_partner","iip_russia","dist","contig","comlang_off","unfriendly","sanction","eu_member","eaes_member","brics_member","cis_member"]]

final.to_csv("data/result/final.csv", index=False)
final

,year,hs2,exporter,importer,exporter_iso3,importer_iso3,partner,partner_iso3,direction,value,...,iip_russia,dist,contig,comlang_off,unfriendly,sanction,eu_member,eaes_member,brics_member,cis_member
0,2017,09,Albania,Russian Federation,ALB,RUS,Albania,ALB,import,10.014,...,106.88,2063.678,0.0,0.0,1,0,0,0,0,0
1,2017,09,Azerbaijan,Russian Federation,AZE,RUS,Azerbaijan,AZE,import,5144.972,...,106.88,1930.747,1.0,0.0,0,0,0,0,0,1
2,2017,09,Argentina,Russian Federation,ARG,RUS,Argentina,ARG,import,2431.309,...,106.88,13505.360,0.0,0.0,0,0,0,0,0,0
3,2017,09,Australia,Russian Federation,AUS,RUS,Australia,AUS,import,5.605,...,106.88,14503.070,0.0,0.0,1,1,0,0,0,0
4,2017,09,Austria,Russian Federation,AUT,RUS,Austria,AUT,import,19119.567,...,106.88,1675.699,0.0,0.0,1,1,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1330,2023,09,Egypt,Russian Federation,EGY,RUS,Egypt,EGY,import,157.478,...,135.97,2905.404,0.0,0.0,0,0,0,0,0,0
1331,2023,09,United Kingdom,Russian Federation,GBR,RUS,United Kingdom,GBR,import,1179.479,...,135.97,2510.880,0.0,0.0,1,1,0,0,0,0
1332,2023,09,United Rep. of Tanzania,Russian Federation,TZA,RUS,United Rep. of Tanzania,TZA,import,5823.626,...,135.97,6968.894,0.0,0.0,0,0,0,0,0,0
1333,2023,09,USA,Russian Federation,USA,RUS,USA,USA,import,8.300,...,135.97,7517.972,0.0,0.0,1,1,0,0,0,0
